# kluster.ai

[kluster.ai](https://kluster.ai) is a inference service that provides access to a variety of high-performance LLMs including Meta's Llama 3.1 and Llama 3.3 models. This notebook goes over how to use LangChain with kluster.ai for chat models.

## Set the Environment API Key
Make sure to get your API key from kluster.ai. You need to [sign up](https://kluster.ai/auth/signup) and create a new API token from your dashboard.

kluster.ai offers a free tier with generous credits to test their models without requiring a credit card.

In [1]:
# get a new token from your kluster.ai dashboard

import os
from getpass import getpass

from langchain_community.chat_models import ChatKlusterAi
from langchain_core.messages import HumanMessage

KLUSTERAI_API_TOKEN = getpass()

# or pass klusterai_api_token parameter to the ChatKlusterAi constructor
os.environ["KLUSTERAI_API_TOKEN"] = KLUSTERAI_API_TOKEN

chat = ChatKlusterAi(model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo")

messages = [
    HumanMessage(
        content="Translate this sentence from English to French. I love programming."
    )
]
chat.invoke(messages)

 ········


ChatKlusterAiException: Error communicating with kluster.ai API: KlusterAi received an invalid payload: {"error":{"message":"body must be object","errorCode":3001,"type":"invalid_request_error","code":null}}

## `ChatKlusterAi` also supports async and streaming functionality:

In [ ]:
from langchain_core.callbacks import StreamingStdOutCallbackHandler

In [ ]:
await chat.agenerate([messages])

In [ ]:
chat = ChatKlusterAi(
    model="klusterai/Meta-Llama-3.1-8B-Instruct-Turbo",
    streaming=True,
    verbose=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)
chat.invoke(messages)

# Tool Calling

kluster.ai supports tool calling functionality with models like Meta-Llama-3.1-405B and Meta-Llama-3.3-70B.

For a complete list of models that support tool calling, please refer to the [kluster.ai documentation](https://kluster.ai).

In [ ]:
import asyncio

from dotenv import find_dotenv, load_dotenv
from langchain_community.chat_models import ChatKlusterAi
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from pydantic import BaseModel

model_name = "klusterai/Meta-Llama-3.3-70B-Instruct-Turbo"

_ = load_dotenv(find_dotenv())


# Langchain tool
@tool
def foo(something):
    """
    Called when foo
    """
    pass


# Pydantic class
class Bar(BaseModel):
    """
    Called when Bar
    """

    pass


llm = ChatKlusterAi(model=model_name)
tools = [foo, Bar]
llm_with_tools = llm.bind_tools(tools)
messages = [
    HumanMessage("Foo and bar, please."),
]

response = llm_with_tools.invoke(messages)
print(response.tool_calls)
# [{'name': 'foo', 'args': {'something': None}, 'id': 'call_Mi4N4wAtW89OlbizFE1aDxDj'}, {'name': 'Bar', 'args': {}, 'id': 'call_daiE0mW454j2O1KVbmET4s2r'}]


async def call_ainvoke():
    result = await llm_with_tools.ainvoke(messages)
    print(result.tool_calls)


# Async call
asyncio.run(call_ainvoke())
# [{'name': 'foo', 'args': {'something': None}, 'id': 'call_ZH7FetmgSot4LHcMU6CEb8tI'}, {'name': 'Bar', 'args': {}, 'id': 'call_2MQhDifAJVoijZEvH8PeFSVB'}]